> **Versión 2 (19-10-2026) · solo descarga del clima**
> - Este cuaderno queda reducido a las celdas 1–5: descarga de Open-Meteo y control de calidad. Producen `EXTERNAS/CLIMA_BOGOTA_PUNTOS_HORARIO.parquet`, que es la entrada de `0160_ClimaAlineado`.
> - Las antiguas celdas 6 y 7 se retiraron (realimentación P5): usaban la lluvia de la **misma hora**, convertían faltantes en 0 (`fillna(0)`) y cruzaban por la hora de **admisión en origen**. Las reemplaza `0160_ClimaAlineado`, que usa solo horas anteriores y cruza con T0 (primera salida a zona urbana).
> - Orden vigente: 0300 v5 (hasta pos. 37) → 0310 y 0320 → decisión A/B → 0330 → **0150 (este)** → 0160 → 0210 → 0410_2 → 0420.

# 0150 · Descarga del clima histórico de Bogotá (insumo de H2)

**Qué hace:** descarga de Open-Meteo (API gratuita, sin clave para uso académico) la lluvia, temperatura y humedad **hora a hora** de enero a agosto de 2026 en 10 puntos de Bogotá y guarda `EXTERNAS/CLIMA_BOGOTA_PUNTOS_HORARIO.parquet`.

**Dónde correrlo:** en Databricks. Si Databricks no tiene salida a internet, correr las celdas en Colab o Jupyter con `PATH_DATABRICKS` apuntando a una carpeta local y subir el Parquet de `EXTERNAS/` al Volume.

**Fuente:** Open-Meteo Historical Weather API (`archive-api.open-meteo.com`), modelo *best match* (IFS + ERA5 / ERA5-Land). Es una **reconstrucción retrospectiva**: no estaba disponible en el momento del despacho. Citarla así en el documento.

In [ ]:
# CELDA 1 · Librerías
%pip install duckdb requests

In [ ]:
# CELDA 2 · Rutas y parámetros
import os, time, datetime
import requests
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATH_DATABRICKS = "/Volumes/proyectogrado/source/sourcedb/"   # o una carpeta local si se corre fuera de Databricks

CARPETA_EXTERNAS        = f"{PATH_DATABRICKS}EXTERNAS/"
RUTA_CLIMA_PUNTOS       = f"{CARPETA_EXTERNAS}CLIMA_BOGOTA_PUNTOS_HORARIO.parquet"   # una fila por punto y hora
RUTA_CLIMA_CIUDAD       = f"{CARPETA_EXTERNAS}CLIMA_BOGOTA_CIUDAD_HORARIO.parquet"   # una fila por hora (promedio ciudad)

FECHA_INICIO = "2026-01-01"
FECHA_FIN    = "2026-08-31"
os.makedirs(CARPETA_EXTERNAS, exist_ok=True)
print("Destino:", CARPETA_EXTERNAS)

In [ ]:
# CELDA 3 · Puntos de referencia en Bogotá (centro aproximado de 10 localidades)
# Cubren norte, occidente, centro, sur y suroriente para captar que la lluvia no cae igual en toda la ciudad.
PUNTOS = pd.DataFrame([
    ("USAQUEN",          4.703, -74.031),
    ("SUBA",             4.741, -74.084),
    ("ENGATIVA",         4.705, -74.113),
    ("FONTIBON",         4.673, -74.146),
    ("CHAPINERO_CENTRO", 4.634, -74.066),
    ("KENNEDY",          4.630, -74.152),
    ("BOSA",             4.618, -74.190),
    ("CIUDAD_BOLIVAR",   4.560, -74.150),
    ("SAN_CRISTOBAL",    4.566, -74.087),
    ("USME",             4.505, -74.113),
], columns=["PUNTO", "LAT", "LON"])
display(PUNTOS)

In [ ]:
# CELDA 4 · Descarga desde Open-Meteo (una sola petición para los 10 puntos)
URL = "https://archive-api.open-meteo.com/v1/archive"
VARIABLES = ["precipitation", "rain", "temperature_2m", "relative_humidity_2m", "weather_code"]

params = {
    "latitude":  ",".join(str(x) for x in PUNTOS["LAT"]),
    "longitude": ",".join(str(x) for x in PUNTOS["LON"]),
    "start_date": FECHA_INICIO,
    "end_date":   FECHA_FIN,
    "hourly": ",".join(VARIABLES),
    "timezone": "America/Bogota",   # misma hora local que G_FECHA_ENVIO
}

def descargar(params, intentos=4):
    for i in range(intentos):
        r = requests.get(URL, params=params, timeout=120)
        if r.status_code == 200:
            return r.json()
        print(f"Intento {i+1}: HTTP {r.status_code} -> {r.text[:200]}")
        time.sleep(10 * (i + 1))
    raise RuntimeError("No se pudo descargar el clima. ¿Databricks tiene salida a internet?")

respuesta = descargar(params)
if isinstance(respuesta, dict):        # con un solo punto la API devuelve un objeto, no una lista
    respuesta = [respuesta]

tablas = []
for punto, bloque in zip(PUNTOS["PUNTO"], respuesta):
    h = pd.DataFrame(bloque["hourly"])
    h["PUNTO"] = punto
    tablas.append(h)
df_puntos = pd.concat(tablas, ignore_index=True)
df_puntos["FECHA_HORA"] = pd.to_datetime(df_puntos["time"])
df_puntos = df_puntos.drop(columns="time").rename(columns={
    "precipitation": "PRECIPITACION_MM", "rain": "LLUVIA_MM", "temperature_2m": "TEMPERATURA_C",
    "relative_humidity_2m": "HUMEDAD_PCT", "weather_code": "CODIGO_CLIMA"})

df_puntos.to_parquet(RUTA_CLIMA_PUNTOS, index=False)
print(f"{len(df_puntos):,} filas ({df_puntos['PUNTO'].nunique()} puntos × {df_puntos['FECHA_HORA'].nunique():,} horas)")
print("Guardado en:", RUTA_CLIMA_PUNTOS)

In [ ]:
# CELDA 5 · Control de calidad y gráfica de lluvia diaria
print("Nulos por variable:"); display(df_puntos.isna().sum())

diaria = (df_puntos.assign(FECHA=df_puntos["FECHA_HORA"].dt.date)
          .groupby(["FECHA", "PUNTO"])["PRECIPITACION_MM"].sum().unstack())
fig, ax = plt.subplots(figsize=(14, 4))
diaria.mean(axis=1).plot(ax=ax)
ax.set_title("Precipitación diaria promedio en Bogotá (mm) · enero–agosto 2026")
ax.set_ylabel("mm/día"); plt.tight_layout(); plt.show()

print("Diferencia entre puntos (lluvia total del periodo, mm):")
display(diaria.sum().sort_values(ascending=False).round(0))

## Siguiente paso
Las variables de clima y su cruce con la base ya no se construyen aquí. Ejecutar `0160_ClimaAlineado`.